# 00 — Setup check

Run this once on **your own** Kaggle (or Colab) account. It confirms you can:
1. clone the repo and install the requirements,
2. find the group audio dataset,
3. load the frozen split and get the **same split hash** as everyone else (`b74d294f…`),
4. load a clip and compute features.

Kaggle settings: Accelerator **GPU T4 x2** (or P100) · Internet **On** · Add Input → `group21-swahili-audio`.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/<org>/nlp-sequential-modeling-group21.git"  # TODO: set after creating the GitHub repo
REPO_DIR = "nlp-sequential-modeling-group21"

IN_NOTEBOOK_CLOUD = os.path.isdir("/kaggle/input") or "google.colab" in sys.modules
if IN_NOTEBOOK_CLOUD:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # running locally from notebooks/

sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

In [ ]:
from src.paths import detect_env, get_data_dir, get_output_dir
from src.data import audio_dir
from src.utils import gpu_name, set_seed

set_seed(42)
print("environment :", detect_env())
print("data dir    :", get_data_dir())
print("audio dir   :", audio_dir())
print("output dir  :", get_output_dir())
print("device      :", gpu_name())

In [ ]:
import json
from src.data import SPLITS_DIR, label_names, load_splits, split_hash, verify_splits

manifest = json.loads((SPLITS_DIR / "manifest.json").read_text(encoding="utf-8"))
splits = load_splits()

print("split hash :", split_hash())
print("matches    :", verify_splits())
print("sizes      :", {k: len(v) for k, v in splits.items()}, "(manifest:", manifest["sizes"], ")")
print("classes    :", label_names())
assert verify_splits(), "Split mismatch - pull the latest main and tell M1"
splits["train"].head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from src.data import load_audio
from src import features

row = splits["train"].iloc[0]
wav = load_audio(row.path)
crop = features.preprocess(wav, crop="energy", seconds=1.5)
mel = features.log_mel(crop)
print(f"{row.id}: '{row.label}' ({row.gloss}), raw {len(wav)/16000:.2f} s -> crop {len(crop)/16000:.2f} s, log-mel {mel.shape}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(np.arange(len(wav)) / 16000, wav, lw=0.5)
axes[0].set(title=f"Raw waveform: {row.label}", xlabel="seconds")
axes[1].imshow(mel.T, origin="lower", aspect="auto", cmap="magma")
axes[1].set(title="Log-mel of the 1.5 s energy window", xlabel="frame (10 ms)", ylabel="mel band")
plt.tight_layout()

✅ If the split cell printed `matches : True` and the plots appear, post your split hash in the group chat and tick *Phase 1 setup* in the tracker.